In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
import pandas as pd
import cyvcf2
from pathlib import Path
import numpy as np

## Filtering only associative variant sites
This is the SBC10-private variant group: SBC10 carries the alt allele and none of SBC4, SBC11, SBC23 do (the low-TAA accession's private variants, candidate causal sites for TAA loss).

In [24]:
!../docker/run.sh bcftools view -Oz -i 'GT[1]="alt" && GT[0]!="alt" && GT[2]!="alt" && GT[3]!="alt"' ../results/combined/all.annotated.vcf.gz -o ../analysis/data/SBC10.vcf.gz
!../docker/run.sh bcftools index -t ../analysis/data/SBC10.vcf.gz

## Parse VCF into pandas dataframe

In [3]:
ANN_FIELDS = [
    "ann_allele", "ann_effect", "ann_impact", "ann_gene_name", "ann_gene_id",
    "ann_feature_type", "ann_feature_id", "ann_biotype", "ann_rank",
    "ann_hgvs_c", "ann_hgvs_p", "ann_cdna_pos", "ann_cds_pos", "ann_aa_pos",
    "ann_distance", "ann_extra",
]

SIFT_FIELDS = [
    "sift_allele", "sift_transcript", "sift_gene_id", "sift_gene_name",
    "sift_region", "sift_variant_type", "sift_aa_change", "sift_aa_pos",
    "sift_score", "sift_median", "sift_num_seqs", "sift_allele_type",
    "sift_prediction",
]

# Sniffles2 INFO keys -> sniffles_-prefixed column names (same nomenclature as ANN/SIFT)
SNIFFLES_KEY_MAP = {
    "SVTYPE": "sniffles_svtype", "SVLEN": "sniffles_svlen", "END": "sniffles_end",
    "SUPPORT": "sniffles_support", "SUPP_VEC": "sniffles_supp_vec",
    "STRAND": "sniffles_strand", "AF": "sniffles_af",
    "STDEV_POS": "sniffles_stdev_pos", "STDEV_LEN": "sniffles_stdev_len",
    "COVERAGE": "sniffles_coverage", "PRECISE": "sniffles_precise",
    "IMPRECISE": "sniffles_imprecise", "MOSAIC": "sniffles_mosaic",
}
SNIFFLES_FIELDS = list(SNIFFLES_KEY_MAP.values())

# Numeric/flag INFO columns to cast from string -> real dtype after parsing
# (dtypes taken from the VCF header: Type=Integer/Float, Type=Flag). Nullable
# Int64/float tolerate the NaNs left by SNP rows (no sniffles_*) and non-coding
# sites (no sift_*). SUPP_VEC/STRAND/COVERAGE stay strings on purpose ("0100"
# leading zeros, "+-", "48,11,..." Number=. list).
INT_COLS   = ["sniffles_svlen", "sniffles_end", "sniffles_support",
              "sift_aa_pos", "sift_num_seqs", "ann_distance"]
FLOAT_COLS = ["sniffles_af", "sniffles_stdev_pos", "sniffles_stdev_len",
              "sift_score", "sift_median"]
FLAG_COLS  = ["sniffles_precise", "sniffles_imprecise", "sniffles_mosaic"]

# SIFT4G writes DELETERIOUS (score < 0.05) or TOLERATED; NA means non-coding
_SIFT_PRIORITY = {"DELETERIOUS": 0, "TOLERATED": 1}
# impact severity for tiebreaking (HIGH most severe -> lowest rank)
_IMPACT_RANK = {"HIGH": 0, "MODERATE": 1, "LOW": 2, "MODIFIER": 3}


def _parse_info_kv(info_str):
    """Split a raw INFO string into a flat dict.
    'KEY=VALUE' -> {KEY: VALUE} (split on first '=' only), bare flags -> {FLAG: True}."""
    kv = {}
    for token in info_str.split(";"):
        if "=" in token:
            k, v = token.split("=", 1)
            kv[k] = v
        elif token:
            kv[token] = True
    return kv


def parse_info(info_str):
    """Parse a raw VCF INFO string into a structured dict.

    Returns is_sv (True for Sniffles2 SV records), ann_raw / sift_raw (the raw
    ANN= / SIFTINFO= strings, or None), and the Sniffles2 fields renamed with a
    sniffles_ prefix (None when absent / non-SV record). ann_raw / sift_raw are
    consumed by parse_vcf to explode them into ann_* / sift_* columns.
    """
    kv = _parse_info_kv(info_str)
    result = {
        "is_sv":    "SVTYPE" in kv,
        "ann_raw":  kv.get("ANN"),
        "sift_raw": kv.get("SIFTINFO"),
    }
    for vcf_key, col in SNIFFLES_KEY_MAP.items():
        result[col] = kv.get(vcf_key)        # None when absent
    return result


def _parse_ann(raw):
    """Split a SnpEff ANN= string into a list of dicts (one per ANN entry)."""
    records = []
    for entry in raw.split(","):
        parts = entry.split("|")
        parts += [""] * (len(ANN_FIELDS) - len(parts))
        records.append(dict(zip(ANN_FIELDS, parts[:len(ANN_FIELDS)])))
    return records


def _parse_siftinfo(raw):
    """Map (allele, transcript) -> best SIFT record from a SIFTINFO= string.

    SnpEff ann_feature_id and SIFT sift_transcript share the XM_* namespace, so
    they join directly. On duplicate keys, DELETERIOUS beats TOLERATED.
    """
    by_key = {}
    for entry in raw.split(","):
        parts = entry.split("|")
        parts += [""] * (len(SIFT_FIELDS) - len(parts))
        d = dict(zip(SIFT_FIELDS, parts[:len(SIFT_FIELDS)]))
        key = (d["sift_allele"], d["sift_transcript"])
        pred = d.get("sift_prediction", "")
        current = by_key.get(key)
        if current is None or _SIFT_PRIORITY.get(pred, 99) < _SIFT_PRIORITY.get(current.get("sift_prediction", ""), 99):
            by_key[key] = d
    return by_key


def _reduce_annotations(annotations, sift_by_key):
    """Reduce a variant's many ANN entries to ONE per gene (see genomics_scoring.py).

    Deterministic sort (lowest key wins): (1) SIFT match on (allele, transcript),
    (2) impact severity HIGH>MODERATE>LOW>MODIFIER, (3) original ANN order,
    (4) protein_coding over other biotypes. Returns (ann, sift) pairs, one per gene.
    """
    by_gene = {}
    for order, ann in enumerate(annotations):
        key = (ann.get("ann_allele", ""), ann.get("ann_feature_id", ""))
        sort_key = (
            0 if key in sift_by_key else 1,
            _IMPACT_RANK.get(ann.get("ann_impact", ""), 99),
            order,
            0 if ann.get("ann_biotype") == "protein_coding" else 1,
        )
        by_gene.setdefault(ann.get("ann_gene_id", ""), []).append((sort_key, key, ann))

    reduced = []
    for entries in by_gene.values():
        _, key, ann = min(entries, key=lambda t: t[0])
        reduced.append((ann, sift_by_key.get(key, {})))
    return reduced


def _fmt_gt(gt):
    """cyvcf2 genotype list -> VCF-style string: [1, 1, False] -> '1/1',
    [0, 1, True] -> '0|1', missing alleles (-1) -> '.'. Handles haploid too."""
    *alleles, phased = gt
    sep = "|" if phased else "/"
    return sep.join("." if a < 0 else str(a) for a in alleles)


def parse_vcf(vcf_path):
    """Parse an annotated VCF into a tidy DataFrame — one row per (variant, gene).

    Per row: chrom, pos, ref, alt, qual, filter; one GT column per sample;
    is_sv and the sniffles_* fields (None for SNP/indels); the SnpEff ANN entry
    exploded into ann_* columns (one per gene via _reduce_annotations); and the
    matching SIFT4G record exploded into sift_* columns. Records without ANN
    (e.g. the ~0.24% of SVs lacking SnpEff annotation) emit a single row with no
    ann_*/sift_* fields. No scoring is applied.

    INFO-derived columns arrive as strings/None; INT_COLS/FLOAT_COLS/FLAG_COLS
    are cast to nullable Int64 / float64 / bool before returning.
    """
    vcf = cyvcf2.VCF(vcf_path)
    samples = vcf.samples
    rows = []
    for v in vcf:
        base = {
            "chrom":  v.CHROM,
            "pos":    v.POS,
            "ref":    v.REF,
            "alt":    ",".join(v.ALT),
            "qual":   v.QUAL,
            "filter": v.FILTER,
        }
        base.update(zip(samples, map(_fmt_gt, v.genotypes)))

        info = parse_info(";".join(
            k if val is True else f"{k}={val}" for k, val in v.INFO
        ))
        ann_raw = info.pop("ann_raw")
        sift_raw = info.pop("sift_raw")
        base.update(info)                    # is_sv + sniffles_* columns

        if not ann_raw:
            rows.append(base)
            continue

        annotations = _parse_ann(ann_raw)
        sift_by_key = _parse_siftinfo(sift_raw) if sift_raw else {}
        for ann, sift in _reduce_annotations(annotations, sift_by_key):
            rows.append({**base, **ann, **sift})

    vcf.close()

    df = pd.DataFrame(rows)
    for col in INT_COLS:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in FLOAT_COLS:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in FLAG_COLS:
        if col in df:
            df[col] = df[col].notna()        # flag present -> True, else False
    
    df["sift_score"] = pd.to_numeric(df.get("sift_score"), errors="coerce")
    df["sift_score_c"] = 1 - df["sift_score"]
    return df

In [4]:
df_variants = parse_vcf("../analysis/data/SBC10.vcf.gz")

In [5]:
df_variants = df_variants[(~df_variants['ann_gene_id'].str.contains("-", regex=False, na=False, case=False)) & ~df_variants['ann_gene_id'].str.contains("&", regex=False, na=False, case=False)]
df_variants

,chrom,pos,ref,alt,qual,filter,SBC4,SBC10,SBC11,SBC23,...,sift_region,sift_variant_type,sift_aa_change,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c
0,NC_012870.2,1619,G,A,54.540001,None,./.,1|0,./.,./.,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
2,NC_012870.2,5784,A,AG,25.700001,None,./.,0/1,./.,./.,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
4,NC_012870.2,5784,A,AGG,25.700001,None,./.,1/0,./.,./.,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
6,NC_012870.2,6934,A,G,31.510000,None,./.,1|0,./.,./.,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
7,NC_012870.2,6934,A,G,31.510000,None,./.,1|0,./.,./.,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740863,NC_008360.1,263767,N,<INV>,54.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
740864,NC_008360.1,263767,N,<INV>,54.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
740866,NC_008360.1,341568,N,<INV>,59.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN
740867,NC_008360.1,341568,N,<INV>,59.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN


## Variant to gene-level annotation
Converting one-row-per-variant data frame into one-row-per-gene:
1. Score each variant's impact (detected by SnpEff and SIFT)
2. Output worst score (maximum) among variants within one gene

In [6]:
IMPACT_BANDS = {
    "MODIFIER": (0.00, 0.25),
    "LOW":      (0.25, 0.50),
    "MODERATE": (0.50, 0.75),
    "HIGH":     (0.75, 1.00),
}

def scoring(df: pd.DataFrame) -> pd.DataFrame:
    lo = df["ann_impact"].map(lambda i: IMPACT_BANDS.get(i, (np.nan, np.nan))[0])
    hi = df["ann_impact"].map(lambda i: IMPACT_BANDS.get(i, (np.nan, np.nan))[1])
    width = hi - lo

    score = lo + 0.5 * width # band midpoint default

    refine = df["ann_impact"].eq("MODERATE") & df["sift_score_c"].notna()
    score = score.mask(refine, lo + df["sift_score_c"] * width) # SIFT within MODERATE only

    df["score"] = score
    return df

def _append_rank(out: pd.DataFrame) -> pd.DataFrame:
    """Insert ``percentile`` and ``rank`` columns immediately right of ``score``.

    ``rank`` is 1 = highest score (dense ranking on ties); ``percentile`` is the
    score's position in [0, 100], higher score -> higher percentile. NaN scores
    (no length/effect match) get NaN rank and percentile. Assumes ``out`` is
    already sorted by score descending.
    """
    percentile = out["score"].rank(ascending=True, pct=True) * 100
    rank = out["score"].rank(ascending=False, method="dense").astype("Int64")
    pos = out.columns.get_loc("score") + 1
    out.insert(pos, "percentile", percentile)
    out.insert(pos + 1, "rank", rank)
    return out

def merge_to_gene_max(df_scored: pd.DataFrame, gene_key: str = "ann_gene_id") -> pd.DataFrame:
    """Collapse variant/transcript rows to one row per gene.

    Maximum / worst-variant approach: each gene's ``score`` is the score of its
    single most damaging variant. Only gene-level columns (GENE_LEVEL_COLS) are
    emitted; variant-level fields are dropped because the output is a per-gene
    product. Rows with no gene id are dropped.
    """
    scored = df_scored[df_scored[gene_key].fillna("").ne("")]
    idx = scored.groupby(gene_key)["score"].idxmax()       # worst variant per gene
    out = scored.loc[idx]
    return _append_rank(out.sort_values("score", ascending=False))

In [7]:
df_variants_scored = scoring(df_variants)
df_variants_scored

,chrom,pos,ref,alt,qual,filter,SBC4,SBC10,SBC11,SBC23,...,sift_variant_type,sift_aa_change,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c,score
0,NC_012870.2,1619,G,A,54.540001,None,./.,1|0,./.,./.,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
2,NC_012870.2,5784,A,AG,25.700001,None,./.,0/1,./.,./.,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
4,NC_012870.2,5784,A,AGG,25.700001,None,./.,1/0,./.,./.,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
6,NC_012870.2,6934,A,G,31.510000,None,./.,1|0,./.,./.,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
7,NC_012870.2,6934,A,G,31.510000,None,./.,1|0,./.,./.,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740863,NC_008360.1,263767,N,<INV>,54.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
740864,NC_008360.1,263767,N,<INV>,54.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
740866,NC_008360.1,341568,N,<INV>,59.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125
740867,NC_008360.1,341568,N,<INV>,59.000000,None,0/0,0/1,0/0,0/0,...,NaN,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125


In [8]:
df_genes = merge_to_gene_max(df_variants_scored)
df_genes

,chrom,pos,ref,alt,qual,filter,SBC4,SBC10,SBC11,SBC23,...,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c,score,percentile,rank
132525,NC_012871.2,27196877,A,G,38.189999,None,./.,0|1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
182463,NC_012871.2,74544928,GGGGGTAGGGCCGTA,G,27.250000,None,./.,1/1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
718762,NC_012879.2,49517611,G,A,52.889999,None,./.,0|1,./.,./.,...,330,NaN,NaN,<NA>,novel,NA,NaN,0.875,97.660525,1
103995,NC_012871.2,6140456,T,A,35.509998,None,./.,1/1,./.,./.,...,1,0.0,3.21,51,novel,DELETERIOUS,1.0,0.875,97.660525,1
106138,NC_012871.2,7093340,N,[NC_012874.2:41159328[N,60.000000,None,0/0,0/1,0/0,0/0,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197923,NC_012872.2,6188409,C,T,26.830000,None,./.,1/1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125,40.772072,101
197895,NC_012872.2,6095270,C,CAT,38.009998,None,./.,1/1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125,40.772072,101
197892,NC_012872.2,6056066,T,TTA,61.990002,None,0/0,1/1,0/0,0/0,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125,40.772072,101
197888,NC_012872.2,6048917,C,T,35.790001,None,./.,1/1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.125,40.772072,101


# DMR analysis

In [9]:
# DMR_annotated.tsv schema (one row per DMR x overlapping feature x gene):
#   chr, start, end, diff.Methy, direction (= "hyper_<accession>"),
#   sample_a, sample_b, feature (promoter|exon|CDS|intron|intergenic), gene_label
DMR_PATH = Path("../results/DMR/DMR_annotated.tsv")

df_dmr = pd.read_csv(DMR_PATH, sep="\t")
print(f"{len(df_dmr):,} DMR-feature rows | features: {sorted(df_dmr['feature'].unique())}")

# --- Candidate genes: promoter HYPOMETHYLATED in the low-TAA accession SBC10 ---
# SBC10 is the lone low-TAA accession. A promoter LESS methylated in SBC10 than
# in a high-TAA accession is de-repressed in SBC10 / silenced in the high-TAA
# ones -- the expression contrast that could underlie the low-vs-high TAA
# split. So candidates are genes whose promoter is hypomethylated in SBC10.
#
# Direction encoding: `direction` names the HYPER accession ("hyper_<acc>").
# Hypomethylated in SBC10  <=>  the contrast involves SBC10 AND the hyper side
# is the OTHER accession, i.e. direction != "hyper_SBC10".
SBC10 = "SBC10"

promoter_dmr = df_dmr[
    (df_dmr["feature"] == "promoter")
    & df_dmr["gene_label"].notna()
    & (df_dmr["gene_label"].astype(str).str.strip() != "")
].copy()

involves_sbc10 = (promoter_dmr["sample_a"] == SBC10) | (promoter_dmr["sample_b"] == SBC10)
hypo_in_sbc10  = involves_sbc10 & (promoter_dmr["direction"] != f"hyper_{SBC10}")
sbc10_hypo = promoter_dmr[hypo_in_sbc10].copy()
sbc10_hypo["abs_diff"] = sbc10_hypo["diff.Methy"].abs()

# one row per gene: its strongest SBC10-hypomethylated promoter DMR
promoter_candidates = (
    sbc10_hypo
    .sort_values("abs_diff", ascending=False)
    .drop_duplicates(subset="gene_label")
    .loc[:, ["gene_label", "diff.Methy", "abs_diff", "direction",
             "sample_a", "sample_b", "chr", "start", "end"]]
    .reset_index(drop=True)
)
print(f"{int(hypo_in_sbc10.sum()):,} promoter DMRs hypomethylated in {SBC10} -> "
      f"{len(promoter_candidates):,} candidate genes")
promoter_candidates.head(20)

152,253 DMR-feature rows | features: ['CDS', 'exon', 'intergenic', 'promoter']
3,091 promoter DMRs hypomethylated in SBC10 -> 2,617 candidate genes


,gene_label,diff.Methy,abs_diff,direction,sample_a,sample_b,chr,start,end
0,LOC8058048,-0.748799,0.748799,hyper_SBC4,SBC10,SBC4,NC_012872.2,58259104,58262211
1,LOC110436238,-0.692811,0.692811,hyper_SBC23,SBC10,SBC23,NC_012875.2,542007,542225
2,LOC110433864,-0.646674,0.646674,hyper_SBC23,SBC10,SBC23,NC_012872.2,58071613,58071928
3,LOC110432936,-0.595732,0.595732,hyper_SBC4,SBC10,SBC4,NC_012871.2,75831277,75831716
4,LOC8083747,-0.590963,0.590963,hyper_SBC23,SBC10,SBC23,NC_012878.2,41580761,41581267
5,LOC8063333,-0.560185,0.560185,hyper_SBC23,SBC10,SBC23,NC_012871.2,66387757,66389692
6,LOC8064193,-0.534160,0.534160,hyper_SBC23,SBC10,SBC23,NC_012878.2,53816849,53817492
7,LOC8068993,-0.527613,0.527613,hyper_SBC23,SBC10,SBC23,NC_012878.2,55936766,55937814
8,LOC8061075,-0.527613,0.527613,hyper_SBC23,SBC10,SBC23,NC_012878.2,55936766,55937814
9,LOC8068380,-0.526789,0.526789,hyper_SBC23,SBC10,SBC23,NC_012879.2,1606812,1607682


## Promoter hypomethylated in SBC10 across the high-vs-low TAA contrasts

In [10]:
# --- Refinement: hypomethylated in SBC10 CONSISTENTLY across low-vs-high TAA --
# Strongest candidates are hypomethylated in SBC10 in EVERY SBC10-vs-high-TAA
# contrast where the gene's promoter appears -- not just one -- so the signal
# tracks the 3-vs-1 phenotype split rather than a single accession pair.
HIGH_TAA = ["SBC4", "SBC11", "SBC23"]

low_vs_high = promoter_dmr[
    ((promoter_dmr["sample_a"] == SBC10) & promoter_dmr["sample_b"].isin(HIGH_TAA))
    | ((promoter_dmr["sample_b"] == SBC10) & promoter_dmr["sample_a"].isin(HIGH_TAA))
].copy()
low_vs_high["hypo_in_low"] = low_vs_high["direction"] != f"hyper_{SBC10}"

consistency = (
    low_vs_high.groupby("gene_label")
    .agg(n_contrasts=("direction", "size"),
         n_hypo_low=("hypo_in_low", "sum"),
         mean_diff=("diff.Methy", "mean"))
)
# keep genes hypomethylated in SBC10 in every low-vs-high contrast they appear in
consistency["all_hypo_low"] = consistency["n_hypo_low"] == consistency["n_contrasts"]

phenotype_candidates = (
    consistency[consistency["all_hypo_low"] & (consistency["n_contrasts"] >= 2)]
    .sort_values("mean_diff", key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
print(f"{len(phenotype_candidates)} genes with a promoter hypomethylated in {SBC10} "
      f"across >=2 low-vs-high TAA contrasts")
phenotype_candidates.head(20)

355 genes with a promoter hypomethylated in SBC10 across >=2 low-vs-high TAA contrasts


,gene_label,n_contrasts,n_hypo_low,mean_diff,all_hypo_low
0,LOC8063333,2,2,-0.373371,True
1,LOC110431956,2,2,-0.342514,True
2,LOC8080517,2,2,-0.304382,True
3,LOC8085864,2,2,-0.304243,True
4,LOC8062471,2,2,-0.300920,True
5,LOC8079764,2,2,-0.295463,True
6,LOC8075921,3,3,-0.286533,True
7,LOC8078876,2,2,-0.286300,True
8,LOC8068526,2,2,-0.286234,True
9,LOC110430333,2,2,-0.272223,True


In [11]:
phenotype_candidates

,gene_label,n_contrasts,n_hypo_low,mean_diff,all_hypo_low
0,LOC8063333,2,2,-0.373371,True
1,LOC110431956,2,2,-0.342514,True
2,LOC8080517,2,2,-0.304382,True
3,LOC8085864,2,2,-0.304243,True
4,LOC8062471,2,2,-0.300920,True
...,...,...,...,...,...
350,LOC8055420,2,2,-0.116131,True
351,LOC8061032,2,2,-0.115992,True
352,LOC8063685,2,2,-0.115939,True
353,LOC110434016,2,2,-0.113591,True


In [12]:
df_genes[df_genes["percentile"] > 50]

,chrom,pos,ref,alt,qual,filter,SBC4,SBC10,SBC11,SBC23,...,sift_aa_pos,sift_score,sift_median,sift_num_seqs,sift_allele_type,sift_prediction,sift_score_c,score,percentile,rank
132525,NC_012871.2,27196877,A,G,38.189999,None,./.,0|1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
182463,NC_012871.2,74544928,GGGGGTAGGGCCGTA,G,27.250000,None,./.,1/1,./.,./.,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
718762,NC_012879.2,49517611,G,A,52.889999,None,./.,0|1,./.,./.,...,330,NaN,NaN,<NA>,novel,NA,NaN,0.875,97.660525,1
103995,NC_012871.2,6140456,T,A,35.509998,None,./.,1/1,./.,./.,...,1,0.0,3.21,51,novel,DELETERIOUS,1.0,0.875,97.660525,1
106138,NC_012871.2,7093340,N,[NC_012874.2:41159328[N,60.000000,None,0/0,0/1,0/0,0/0,...,<NA>,NaN,NaN,<NA>,NaN,NaN,NaN,0.875,97.660525,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16510,NC_012870.2,13777173,C,T,28.809999,None,./.,1/1,./.,./.,...,352,NaN,NaN,<NA>,novel,NA,NaN,0.375,83.858537,100
514765,NC_012876.2,57211659,C,T,33.889999,None,./.,1/1,./.,./.,...,331,1.0,2.68,89,novel,TOLERATED,0.0,0.375,83.858537,100
2779,NC_012870.2,1559623,C,T,63.220001,None,./.,1/1,./.,./.,...,351,1.0,2.61,300,novel,TOLERATED,0.0,0.375,83.858537,100
518993,NC_012876.2,59393594,G,T,27.070000,None,./.,1/1,./.,./.,...,94,1.0,2.58,102,novel,TOLERATED,0.0,0.375,83.858537,100


In [13]:
print(len(df_genes[df_genes["ann_impact"] == "HIGH"]))
print(len(df_genes[df_genes["ann_impact"] == "MODERATE"]))


1027
2005


# Refining the causative candidate genes list
Aconitate isomerase is:
- Isomerase gene, showing isomerase activity
- Expressed under phytochrome regulation

## Respiratory Genes Expressed under light condition
From Igamberdiev et al. (2014)
These genes are thought to be co-expressed with aconitate isomerase. Only mitochondrial genes are selected:
- 8071625 (citrate synthase)
- 8054957 & 8083786 (malate dehydrogenase)

citrate synthase
110433903
8068084
8071625
8073811
8086010

malate dh
110430391
110436944
110436993
8054957
8055458
8055966
8057802
8065238
8070482
8079364
8083786

high co-expression (>=1.25) genes module (mix citrate synthase - malate dh)
8054957
8083786
8086010
8070482
110433903
8071625
8055458

isomerase
`analysis/data/sorghum_isomerase.txt`

In [14]:
# find which isomerase is highly co-expressed with the light-respiration gene module

# ATTED-II Sbi-r.c1-0 (v25-12, subagging z-transformed Mutual Rank): one file per
# gene under ATTED_ZD, each a two-column <partner_EGI>\t<coex_score> table sorted
# descending by score. The z-transform standardizes MR onto a comparable scale
# across different bait genes, which is what makes averaging across the 7-gene
# module below meaningful.
ATTED_ZD = Path("/mnt/hdd/daffa/Sbi-r.c1-0/Sbi-r.v25-12.G21627-S807.combat_pca.subagging.z.d")

MODULE_GENES = ["8054957", "8083786", "8086010", "8070482", "110433903", "8071625", "8055458"]

isomerase_genes = [l.strip() for l in open("../analysis/data/sorghum_isomerase.txt") if l.strip()]

def load_neighborhood(egi):
    """partner_EGI -> (coex_score, 1-based rank), plus N_total partners in this bait's file."""
    neigh = {}
    with open(ATTED_ZD / egi) as f:
        for i, line in enumerate(f, start=1):
            partner, score = line.rstrip("\n").split("\t")
            neigh[partner] = (float(score), i)
    return neigh, i

module_neigh = {g: load_neighborhood(g) for g in MODULE_GENES}

rows = []
for c in isomerase_genes:
    scores, ranks = [], []
    for neigh, n_total in module_neigh.values():
        if c in neigh:
            s, r = neigh[c]
        else:
            s, r = np.nan, n_total + 1  # not in ATTED-II's assayed gene set -> worst rank
        scores.append(s)
        ranks.append(r)
    present = [s for s in scores if not np.isnan(s)]
    rows.append({
        "EGI": c,
        "n_hits": len(present),          # out of 7 module genes; 0 if candidate wasn't profiled by ATTED-II
        "mean_score": np.mean(present) if present else np.nan,
        "max_score": max(present) if present else np.nan,
        "mean_rank": np.mean(ranks),
        "min_rank": min(ranks),
    })

df_iso_coex = pd.DataFrame(rows)

gene_info = pd.read_csv("../resources/NCBI_FTP/gene_info_4558", sep="\t", dtype=str)[["GeneID", "description"]]
df_iso_coex = df_iso_coex.merge(gene_info, left_on="EGI", right_on="GeneID", how="left").drop(columns="GeneID")
df_iso_coex = df_iso_coex.sort_values("mean_score", ascending=False).reset_index(drop=True)

print(f"{len(df_iso_coex)} isomerase candidates ranked "
      f"({(df_iso_coex['n_hits'] == 7).sum()} profiled by ATTED-II, "
      f"{(df_iso_coex['n_hits'] == 0).sum()} not in the assayed gene set)")
df_iso_coex

461 isomerase candidates ranked (404 profiled by ATTED-II, 57 not in the assayed gene set)


,EGI,n_hits,mean_score,max_score,mean_rank,min_rank,description
0,8063383,7,3.690571,5.1710,489.571429,42,"phosphoglucomutase, cytoplasmic 2"
1,8060801,7,3.461229,5.3494,1076.571429,9,"triosephosphate isomerase, cytosolic"
2,8076014,7,3.069657,4.4412,542.571429,201,protein disulfide isomerase-like 2-1
3,8061827,7,3.044214,4.1493,618.571429,111,26S protease regulatory subunit 7A
4,8082699,7,2.916757,4.4088,857.142857,92,26S protease regulatory subunit 7A
...,...,...,...,...,...,...,...
456,110435861,0,NaN,NaN,21627.000000,21627,uncharacterized LOC110435861
457,110435897,0,NaN,NaN,21627.000000,21627,ATP-dependent DNA helicase PIF1-like
458,110436500,0,NaN,NaN,21627.000000,21627,probable helicase MAGATAMA 3
459,110437264,0,NaN,NaN,21627.000000,21627,ATP-dependent DNA helicase PIF1-like


## Mutual Rank table: each isomerase candidate vs each of the 7 module genes

Same 461 candidates and 7-gene module as above, but reported per-bait instead of
aggregated: for every (isomerase candidate, module gene) pair, the **Mutual Rank**
`MR = sqrt(r_fwd * r_rev)`, where `r_fwd` is the candidate's rank within the module
gene's ATTED-II neighborhood and `r_rev` is the module gene's rank within the
candidate's own neighborhood (lower MR = tighter reciprocated co-expression).
Candidates absent from ATTED-II's assayed gene set get `NaN` in every MR column.

The summary column uses the arithmetic mean of the raw z-scores (not geometric
mean — the z-transformed co-expression values go negative for weakly/anti
co-expressed pairs, so a geometric mean is undefined there; z-scores are additive
by construction anyway, so arithmetic mean is the natural aggregate).

In [15]:
def reverse_rank(egi, targets):
    """Rank of each gene in `targets` within egi's own ATTED-II neighborhood file.
    Returns {} if egi has no file (not in ATTED-II's assayed gene set)."""
    found = {}
    fpath = ATTED_ZD / egi
    if not fpath.exists():
        return found
    remaining = set(targets)
    with open(fpath) as f:
        for i, line in enumerate(f, start=1):
            partner, _ = line.rstrip("\n").split("\t")
            if partner in remaining:
                found[partner] = i
                remaining.discard(partner)
                if not remaining:
                    break
    return found

module_set = set(MODULE_GENES)

mr_rows = []
for c in isomerase_genes:
    rev = reverse_rank(c, module_set)
    row = {"EGI": c}
    scores = []
    for b in MODULE_GENES:
        neigh, _ = module_neigh[b]
        if c in neigh and b in rev:
            s, r_fwd = neigh[c]
            r_rev = rev[b]
            row[f"MR_{b}"] = np.sqrt(r_fwd * r_rev)
            scores.append(s)
        else:
            row[f"MR_{b}"] = np.nan
    row["mean_coex_score"] = np.mean(scores) if scores else np.nan
    row["n_hits"] = len(scores)
    mr_rows.append(row)

df_mr = pd.DataFrame(mr_rows)
df_mr = df_mr.merge(gene_info, left_on="EGI", right_on="GeneID", how="left").drop(columns="GeneID")
df_mr = df_mr.sort_values("mean_coex_score", ascending=False).reset_index(drop=True)
df_mr

,EGI,MR_8054957,MR_8083786,MR_8086010,MR_8070482,MR_110433903,MR_8071625,MR_8055458,mean_coex_score,n_hits,description
0,8063383,134.424700,60.332413,499.715919,34.292856,70.213959,247.834622,2482.625223,3.690571,7,"phosphoglucomutase, cytoplasmic 2"
1,8060801,143.192179,378.740016,190.677739,32.939338,1382.996746,25.278449,5373.718545,3.461229,7,"triosephosphate isomerase, cytosolic"
2,8076014,126.237871,126.806940,1336.694056,971.406197,908.549393,417.258912,338.876084,3.069657,7,protein disulfide isomerase-like 2-1
3,8061827,989.890903,300.502912,2536.777483,189.420168,222.499438,618.727727,952.839966,3.044214,7,26S protease regulatory subunit 7A
4,8082699,1173.030264,295.276142,3274.320082,136.062486,182.493836,754.102115,1716.044871,2.916757,7,26S protease regulatory subunit 7A
...,...,...,...,...,...,...,...,...,...,...,...
456,110435861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,uncharacterized LOC110435861
457,110435897,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ATP-dependent DNA helicase PIF1-like
458,110436500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,probable helicase MAGATAMA 3
459,110437264,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,ATP-dependent DNA helicase PIF1-like


## Same table, aggregated with Robust Rank Aggregation instead of the arithmetic mean

[Kolde et al. 2012](https://doi.org/10.1093/bioinformatics/btr709)'s RRA treats
each of the 7 module genes as a "voter" that ranks the isomerase candidates by
co-expression strength, then gives each candidate a p-value for how much better
it is ranked, across all 7 voters, than expected under a null of random
ordering (order statistics on normalized ranks, Bonferroni-corrected). Lower
`rra_score` = more consistently top-ranked across the whole module, not just
the 1–2 strongest baits (which the arithmetic mean above can't distinguish).

Two dependency dead ends before landing on the tool actually used here:
- **rpy2 + R's `RobustRankAggreg`**: rpy2 failed to build — its C extension
  links against the host's system R (`/usr/lib64/R`), and conda's linker
  (`compiler_compat/ld`) can't resolve that R build's system-library symbols
  (`libgssapi_krb5`, `readline`, BLAS/LAPACK, `libicuuc`, ...). Not worth
  fighting for one function; would need a conda-provided R + rpy2 instead.
- **pyflagr (current release)**: installs cleanly, but pyflagr 1.0.21 *and*
  1.0.18 silently return an **empty** result for every aggregation method,
  including on FLAGR's own reference test data — a known, unresolved upstream
  bug ([FLAGR issue #4](https://github.com/lakritidis/FLAGR/issues/4)).
  **`pyflagr==1.0.14` is the last version confirmed to work** (pinned below).
  Its output CSV is also quirky — a malformed 4-name header for 5-field rows,
  plus one bogus duplicate-header data row — so it's parsed by hand rather
  than through pyflagr's own (broken, for this version too) `get_output()`.

RRA needs an actual ranked list per voter, so it only covers the 404
candidates ATTED-II assayed; the 57 unassayed candidates get `NaN`, same as
everywhere else in this notebook.

In [16]:
import glob
import os
import shutil

import pyflagr.RRA as RRA  # pip install pyflagr==1.0.14 -- see markdown above for why this exact version

RRA_WORKDIR = os.environ.get("CLAUDE_JOB_DIR", "/tmp") + "/tmp/rra_work"


def run_rra(lists_by_voter, eval_pts=10):
    """Robust Rank Aggregation over ``lists_by_voter = {voter: [item, ...]}``
    (each list ranked best-first, same item set across voters). Returns a
    DataFrame with columns ItemID, rra_rank, rra_score (ascending score = more
    consistently top-ranked across voters).
    """
    rows = []
    for voter, order in lists_by_voter.items():
        n = len(order)
        for rank, item in enumerate(order, start=1):
            rows.append(["Q", voter, item, (n - rank + 1) / n, "run"])
    df_in = pd.DataFrame(rows, columns=["Query", "Voter", "ItemID", "Score", "Label"])

    outdir = os.path.join(RRA_WORKDIR, f"w{os.getpid()}_{id(lists_by_voter)}")
    os.makedirs(outdir, exist_ok=True)
    try:
        RRA.RRA(eval_pts=eval_pts, exact=False).aggregate(input_df=df_in, out_dir=outdir)
        out_file = glob.glob(os.path.join(outdir, "out_*.csv"))[0]
        # pyflagr 1.0.14 writes a malformed header (4 names for 5-field rows) plus
        # one bogus duplicate-header data row; skip both, name the 5 real columns.
        raw = pd.read_csv(out_file, header=None, skiprows=2,
                           names=["Query", "Aggregator", "ItemID", "rra_rank", "rra_score"])
    finally:
        shutil.rmtree(outdir, ignore_errors=True)
    return raw[["ItemID", "rra_rank", "rra_score"]]


def build_mr_table_rra():
    """Identical to ``df_mr`` above (same MR_<bait> columns, same reverse_rank /
    module_neigh machinery), but the summary column is the Robust Rank
    Aggregation p-value instead of the arithmetic mean of z-scores.
    """
    assayed = [c for c in isomerase_genes if c in module_neigh[MODULE_GENES[0]][0]]

    voter_lists = {}
    for b in MODULE_GENES:
        neigh, _ = module_neigh[b]
        voter_lists[b] = sorted(assayed, key=lambda c: neigh[c][0], reverse=True)  # best z-score first
    rra_out = run_rra(voter_lists).astype({"ItemID": str}).set_index("ItemID")

    rows = []
    for c in isomerase_genes:
        rev = reverse_rank(c, module_set)
        row = {"EGI": c}
        for b in MODULE_GENES:
            neigh, _ = module_neigh[b]
            if c in neigh and b in rev:
                s, r_fwd = neigh[c]
                row[f"MR_{b}"] = np.sqrt(r_fwd * rev[b])
            else:
                row[f"MR_{b}"] = np.nan
        if c in rra_out.index:
            row["rra_rank"] = rra_out.loc[c, "rra_rank"]
            row["rra_score"] = rra_out.loc[c, "rra_score"]
        else:
            row["rra_rank"] = np.nan
            row["rra_score"] = np.nan
        rows.append(row)

    out = pd.DataFrame(rows)
    out = out.merge(gene_info, left_on="EGI", right_on="GeneID", how="left").drop(columns="GeneID")
    return out.sort_values("rra_score").reset_index(drop=True)


df_mr_rra = build_mr_table_rra()
df_mr_rra

Processing Query 1 / 1... [  OK ]


,EGI,MR_8054957,MR_8083786,MR_8086010,MR_8070482,MR_110433903,MR_8071625,MR_8055458,rra_rank,rra_score,description
0,8054237,4845.605844,3894.473007,7278.854855,8063.493164,11411.499726,830.344507,3723.008461,NaN,NaN,peptidyl-prolyl cis-trans isomerase CYP65
1,8054265,13273.924439,8444.710179,12516.266216,656.999239,5170.555870,6361.468463,17047.273976,NaN,NaN,probable pre-mRNA-splicing factor ATP-dependen...
2,8054392,19652.124720,11080.291693,19804.825119,13507.999852,20367.650331,15805.038058,13260.787156,NaN,NaN,DEAD-box ATP-dependent RNA helicase 25
3,8054550,1767.692564,6275.714382,4779.655845,12800.874892,3634.804672,5551.075031,978.273990,NaN,NaN,UDP-glucose 4-epimerase 4
4,8054630,4931.939477,5333.124225,2075.456576,6258.458277,7301.706787,18051.505367,7388.961361,NaN,NaN,uncharacterized isomerase BH0283
...,...,...,...,...,...,...,...,...,...,...,...
456,110436943,7442.996708,14549.064781,5903.163813,2584.735190,19887.627209,5003.871301,9168.992202,NaN,NaN,achilleol B synthase-like
457,110437006,6960.505154,6007.612171,2001.284837,660.626975,12733.395855,7631.100838,9859.534522,NaN,NaN,70 kDa peptidyl-prolyl isomerase
458,110437176,11292.259384,19642.699916,13175.553271,21378.882104,21401.941594,18378.091740,17915.257017,NaN,NaN,kinesin-like protein KIN-1
459,110437264,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ATP-dependent DNA helicase PIF1-like
